# Notebook-legal.com.br

[Dataset](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

In [ ]:
%pip install pandas duckdb pyarrow

In [ ]:
import pandas as pd
import duckdb
import pyarrow
from time import perf_counter

In [ ]:
arquivo = "yellow_tripdata_2026-01.parquet"
df = pd.read_parquet(arquivo)
con = duckdb.connect()
con.execute(f"""
    CREATE TABLE corridas AS
    SELECT * FROM read_parquet ('{ arquivo }')
""")

In [ ]:
print(f"Número de Linhas no df: {len(df)}")

con.sql("SELECT COUNT(*) FROM corridas").show()

print(f"Colunas: {df.columns}")

con.sql("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_name = 'corridas'
""").show()

In [ ]:
df = df[(df['tpep_pickup_datetime'] >= '2026-01-01') 
      & (df['tpep_dropoff_datetime'] < '2026-02-01')]

con.sql("""
    DELETE FROM corridas
    WHERE tpep_pickup_datetime < '2026-01-01'
    OR tpep_dropoff_datetime >= '2026-02-01';
""")

In [ ]:
print(f"Número de Linhas no df: {len(df)}")
con.sql("SELECT COUNT(*) FROM corridas").show()

# 1) - Corridas por horário:
Para cada hora de início da corrida (0 a 23, a partir de tpep_pickup_datetime), calcule o número de corridas, a distância média (trip_distance) e o valor médio pago (total_amount). Ordene pelo número de corridas, do maior para o menor. Utilize os registros de janeiro de 2026.

### Pandas

In [ ]:
tempo_pandas1 = perf_counter()
df_ = df.copy()
df_['hour'] = df_['tpep_pickup_datetime'].dt.hour
pandas_results = (
        df_.groupby("hour", as_index=False)
           .agg(
                   n_corridas=('hour', 'size'),
                   mean_distance=('trip_distance', 'mean'),
                   mean_price=('total_amount', 'mean')
           )
           .sort_values('n_corridas', ascending=False)
)
tempo_pandas1 = perf_counter() - tempo_pandas1
print(f'\n\nPANDAS -> {tempo_pandas1:.5f}s')
print(pandas_results)

### DuckDB

In [ ]:
tempo_duckdb1 = perf_counter()
results = con.sql("""
        SELECT date_part('hour', tpep_pickup_datetime) AS hour_pickup, COUNT(*) AS n_corridas, MEAN(trip_distance), MEAN(total_amount)
        FROM corridas
        GROUP BY hour_pickup
        ORDER BY n_corridas DESC
        """).fetchall()
tempo_duckdb1 = perf_counter() - tempo_duckdb1

print(f'DUCKDB -> {tempo_duckdb1:.5f}s')
print('(hour, #, mean_distance, mean_price)')
for result in results:
        print(result)

### Resultados

In [ ]:
print(f'{('DuckDB' if tempo_duckdb1 < tempo_pandas1 else 'Pandas')} foi mais rápido {(tempo_pandas1/tempo_duckdb1) if tempo_duckdb1 < tempo_pandas1 else (tempo_duckdb1/tempo_pandas1):.2f}x')

# 2) - Corridas semelhantes:

Encontre pares de corridas distintas que tenham a mesma origem (PULocationID), o mesmo destino (DOLocationID) e horários de início (tpep_pickup_datetime) com diferença de, no máximo, cinco minutos. Conte cada par apenas uma vez e apresente somente a quantidade total de pares. Para limitar o volume, considere as corridas iniciadas em 15/01/2026, das 8h às 9h (sem incluir 9h).

### Pandas

In [ ]:
tempo_pandas2 = perf_counter()

df_copy = df[
    (df['tpep_pickup_datetime'] >= '2026-01-15 08:00:00') &
    (df['tpep_pickup_datetime'] <  '2026-01-15 09:00:00')
]

tolerance = pd.Timedelta(minutes=5)

df_copy = df_copy.sort_values(['PULocationID', 'DOLocationID', 'tpep_pickup_datetime'])
PUtime = df_copy.groupby(['PULocationID', 'DOLocationID'])['tpep_pickup_datetime']

# Tempo até a corrida mais próxima no mesmo trajeto
df_copy['nearest_race_time'] = pd.concat(
    [PUtime.diff(), PUtime.diff(-1).abs()], axis=1
).min(axis=1)

similar_races_pd = df_copy[df_copy['nearest_race_time'] <= tolerance]

tempo_pandas2 = perf_counter() - tempo_pandas2
print(f'PANDAS -> {tempo_pandas2:.5f}s\n')

print(similar_races_pd[['PULocationID', 'DOLocationID', 'tpep_pickup_datetime', 'nearest_race_time']])

### DuckDB

In [ ]:
tempo_duckdb2 = perf_counter()

similar_races_sql = con.sql("""
  WITH c AS (
    SELECT rowid AS id, 
           PULocationID, 
           DOLocationID, 
           tpep_pickup_datetime AS ts
    FROM corridas
    WHERE tpep_pickup_datetime >= TIMESTAMP '2026-01-15 08:00:00'
      AND tpep_pickup_datetime <  TIMESTAMP '2026-01-15 09:00:00'
  )
  SELECT a.PULocationID, 
         a.DOLocationID, 
         a.ts AS tpep_pickup_datetime,
         MIN(abs(date_diff('second', a.ts, b.ts))) AS nearest_race_seconds
  FROM c a
  JOIN c b
    ON a.id <> b.id
    AND  a.PULocationID = b.PULocationID
    AND a.DOLocationID = b.DOLocationID
    AND b.ts BETWEEN a.ts - INTERVAL 5 MINUTE AND a.ts + INTERVAL 5 MINUTE
  GROUP BY a.id, a.PULocationID, a.DOLocationID, a.ts
  ORDER BY PULocationID, DOLocationID, tpep_pickup_datetime;
  """).df()

tempo_duckdb2 = perf_counter() - tempo_duckdb2
print(f'DUCKDB -> {tempo_duckdb2:.5f}s\n')

print(similar_races_sql)

### Resultados

In [ ]:
print(f'{('DuckDB' if tempo_duckdb2 < tempo_pandas2 else 'Pandas')} foi mais rápido {(tempo_pandas2/tempo_duckdb2) if tempo_duckdb2 < tempo_pandas2 else (tempo_duckdb2/tempo_pandas2):.2f}x')

# 3) Consulta Livre - Horário mais lentos
Formule uma pergunta analítica sobre os dados, diferente das anteriores, e resolva-a em Pandas e SQL sem auxílio de IA. Escreva a pergunta antes das implementações.

### Pandas

In [ ]:
tempo_pandas3 = perf_counter()

df_copy = df.copy()

valid = df_copy["trip_distance"].gt(0) & df_copy["tpep_dropoff_datetime"].gt(df_copy["tpep_pickup_datetime"])
df_copy = df_copy[valid].copy()

df_copy["hour"] = df_copy["tpep_pickup_datetime"].dt.hour
df_copy["duration_min"] = (df_copy["tpep_dropoff_datetime"] - df_copy["tpep_pickup_datetime"]).dt.total_seconds() / 60

by_hour = df_copy.groupby("hour")
trip_count = by_hour.size()
dist_total = by_hour["trip_distance"].sum()
min_total = by_hour["duration_min"].sum()

result = pd.DataFrame({
    "trip_count": trip_count,
    "km_per_min": dist_total / min_total,
}).reset_index()

tempo_pandas3 = perf_counter() - tempo_pandas3
print(f'PANDAS -> {tempo_pandas3:.5f}s\n')

print(result)

### DuckDB

In [ ]:
tempo_duckdb3 = perf_counter()

free_query_sql = con.sql("""
    SELECT
        EXTRACT(hour FROM tpep_pickup_datetime) AS hour,
        COUNT(*) AS trip_count,
        SUM(trip_distance) / (SUM(date_diff('second', tpep_pickup_datetime, tpep_dropoff_datetime)) / 60.0) AS km_per_min
    FROM corridas
    WHERE trip_distance > 0
     AND tpep_dropoff_datetime > tpep_pickup_datetime
    GROUP BY hour
    ORDER BY hour
""").df()

tempo_duckdb3 = perf_counter() - tempo_duckdb3
print(f'DUCKDB -> {tempo_duckdb3:.5f}s\n')

print(free_query_sql)

### Resultados

In [ ]:
print(f'{('DuckDB' if tempo_duckdb3 < tempo_pandas3 else 'Pandas')} foi mais rápido {(tempo_pandas3/tempo_duckdb3) if tempo_duckdb3 < tempo_pandas3 else (tempo_duckdb3/tempo_pandas3):.2f}x')

# 4) Consulta livre
Utilize IA para propor uma pergunta analítica diferente das anteriores e auxiliar na implementação em Pandas e SQL. Informe a pergunta escolhida e registre os principais prompts utilizados.

### Pandas

In [ ]:
def consulta4_pandas(df):
    d = df[
        (df["tpep_pickup_datetime"] >= "2026-01-01")
        & (df["tpep_pickup_datetime"] < "2026-02-01")
        & (df["payment_type"] == 1)
        & (df["fare_amount"] > 0)
        & (df["trip_distance"] > 0)
    ].copy()

    d["faixa_distancia"] = pd.cut(
        d["trip_distance"],
        bins=[0, 2, 5, float("inf")],
        labels=["curta", "media", "longa"],
        right=False,
    )
    d["pct_gorjeta"] = d["tip_amount"] / d["fare_amount"] * 100

    res = (
        d.groupby("faixa_distancia", observed=True)
        .agg(
            n_corridas=("tip_amount", "size"),
            gorjeta_media=("tip_amount", "mean"),
            pct_gorjeta_medio=("pct_gorjeta", "mean"),
        )
        .round(2)
        .reset_index()
        .sort_values("faixa_distancia")
        .reset_index(drop=True)
    )
    res["faixa_distancia"] = res["faixa_distancia"].astype(str)
    return res

tempo_pandas4 = perf_counter()
res_pd = consulta4_pandas(df)

tempo_pandas4 = perf_counter() - tempo_pandas4
print(f'PANDAS -> {tempo_pandas4:.5f}s\n')

print(res_pd)

### DuckDB

In [ ]:
SQL_Q4 = """
SELECT
    CASE
        WHEN trip_distance < 2 THEN 'curta'
        WHEN trip_distance < 5 THEN 'media'
        ELSE 'longa'
    END AS faixa_distancia,
    COUNT(*) AS n_corridas,
    ROUND(AVG(tip_amount), 2) AS gorjeta_media,
    ROUND(AVG(tip_amount / fare_amount * 100), 2) AS pct_gorjeta_medio
FROM corridas
WHERE tpep_pickup_datetime >= '2026-01-01'
  AND tpep_pickup_datetime <  '2026-02-01'
  AND payment_type = 1
  AND fare_amount > 0
  AND trip_distance > 0
GROUP BY faixa_distancia
ORDER BY CASE faixa_distancia
             WHEN 'curta' THEN 1
             WHEN 'media' THEN 2
             ELSE 3
         END
"""

tempo_duckdb4 = perf_counter()
res_sql = con.execute(SQL_Q4).df()

tempo_duckdb4 = perf_counter() - tempo_duckdb4
print(f'DUCKDB -> {tempo_duckdb4:.5f}s\n')

print(res_sql)

### Resultados

In [ ]:
print(f'\n\n{('DuckDB' if tempo_duckdb4 < tempo_pandas4 else 'Pandas')} foi mais rápido {(tempo_pandas4/tempo_duckdb4) if tempo_duckdb4 < tempo_pandas4 else (tempo_duckdb4/tempo_pandas4):.2f}x')

# 5) Corridas simultânea
Considere as corridas iniciadas no dia 15/01/2026 a partir das 8h. Conte quantos pares de corridas tiveram seus períodos de viagem sobrepostos, considerando os horários de início (tpep_pickup_datetime) e de término (tpep_dropoff_datetime). Considere apenas pares de corridas distintas e conte cada par uma única vez. Execute a consulta para três janelas de tempo, das 8h às 9h, das 8h às 10h e das 8h às 12h (sem incluir o horário final). Informe a quantidade de pares e o tempo de execução de cada janela. Se alguma execução exceder a memória disponível, registre isso e não aumente a janela.

### Pandas

In [ ]:
tempo_pandas5 = perf_counter()

inicio_execucao = perf_counter()

mascara = (
    (df["tpep_pickup_datetime"] >= "2026-01-15 08:00:00") &
    (df["tpep_pickup_datetime"] < f"2026-01-15 09:00:00")
)
corridas = df.loc[mascara, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]].sort_values("tpep_pickup_datetime").reset_index(drop=True)

inicios = corridas["tpep_pickup_datetime"].to_numpy()
finais = corridas["tpep_dropoff_datetime"].to_numpy()
total = 0
n = len(corridas)

for i in range(n):
    total += ((inicios[i] < finais[i + 1:]) & (inicios[i + 1:] < finais[i])).sum()

print(f"Pandas: 08h–09h | Corridas: {n} | Pares sobrepostos: {total} | Tempo: {perf_counter() - inicio_execucao:.5f}s")


inicio_execucao = perf_counter()

mascara = (
    (df["tpep_pickup_datetime"] >= "2026-01-15 08:00:00") &
    (df["tpep_pickup_datetime"] < f"2026-01-15 10:00:00")
)
corridas = df.loc[mascara, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]].sort_values("tpep_pickup_datetime").reset_index(drop=True)

inicios = corridas["tpep_pickup_datetime"].to_numpy()
finais = corridas["tpep_dropoff_datetime"].to_numpy()
total = 0
n = len(corridas)

for i in range(n):
    total += ((inicios[i] < finais[i + 1:]) & (inicios[i + 1:] < finais[i])).sum()

print(f"Pandas: 08h–10h | Corridas: {n} | Pares sobrepostos: {total} | Tempo: {perf_counter() - inicio_execucao:.5f}s")


inicio_execucao = perf_counter()

mascara = (
    (df["tpep_pickup_datetime"] >= "2026-01-15 08:00:00") &
    (df["tpep_pickup_datetime"] < f"2026-01-15 12:00:00")
)
corridas = df.loc[mascara, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]].sort_values("tpep_pickup_datetime").reset_index(drop=True)

inicios = corridas["tpep_pickup_datetime"].to_numpy()
finais = corridas["tpep_dropoff_datetime"].to_numpy()
total = 0
n = len(corridas)

for i in range(n):
    total += ((inicios[i] < finais[i + 1:]) & (inicios[i + 1:] < finais[i])).sum()

print(f"Pandas: 08-12h | Corridas: {n} | Pares sobrepostos: {total} | Tempo: {perf_counter() - inicio_execucao:.5f}s")

tempo_pandas5 = perf_counter() - tempo_pandas5
print(f'\nPANDAS -> {tempo_pandas5:.5f}s')

### DuckDB

In [ ]:
def sobreposicoes_duckdb(inicio, fim, rotulo):
    t0 = perf_counter()

    query = f"""
    WITH janela AS (
        SELECT
            tpep_pickup_datetime  AS inicio,
            tpep_dropoff_datetime AS fim,
            ROW_NUMBER() OVER (ORDER BY tpep_pickup_datetime) AS id
        FROM corridas
        WHERE tpep_pickup_datetime >= TIMESTAMP '{inicio}'
          AND tpep_pickup_datetime <  TIMESTAMP '{fim}'
    )
    SELECT
        (SELECT COUNT(*) FROM janela) AS n,
        COUNT(*) AS total
    FROM janela a
    JOIN janela b
      ON  a.id < b.id
      AND a.inicio < b.fim
      AND b.inicio < a.fim
    """

    n, total = con.sql(query).fetchone()
    print(f"DuckDB: {rotulo} | Corridas: {n} | Pares sobrepostos: {total} | Tempo: {perf_counter() - t0:.5f}s")

tempo_duckdb5 = perf_counter()

sobreposicoes_duckdb("2026-01-15 08:00:00", "2026-01-15 09:00:00", "08h-09h")
sobreposicoes_duckdb("2026-01-15 08:00:00", "2026-01-15 10:00:00", "08h-10h")
sobreposicoes_duckdb("2026-01-15 08:00:00", "2026-01-15 12:00:00", "08h-12h")

tempo_duckdb5 = perf_counter() - tempo_duckdb5
print(f'\nDUCKDB -> {tempo_duckdb5:.5f}s')

### Resultados

In [ ]:
print(f'\n\n{('DuckDB' if tempo_duckdb5 < tempo_pandas5 else 'Pandas')} foi mais rápido {(tempo_pandas5/tempo_duckdb5) if tempo_duckdb5 < tempo_pandas5 else (tempo_duckdb5/tempo_pandas5):.2f}x')